In [1]:
!nvidia-smi

Wed Nov 11 19:35:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
import sys

os.environ["TL_PYTHON"] = sys.executable
%env TL_SOURCE_ARCHIVE=/content/source-upload.zip
%env TL_DATA_ARCHIVE=/content/drive/MyDrive/cv-practice/eurosat_rgb.zip
%env TL_DATA_SHA256=d53b32fa50a695af988548b019ab0ff0317e0c2066d54d7671b16b179da66bb8
%env TL_PROJECT_DIR=/content/cv-practice
%env TL_DATA_DIR=/content/data/EuroSAT_RGB
%env TL_OUTPUT_DIR=/content/eurosat-run-01
%env TL_DATASET_VERSION=1

env: TL_SOURCE_ARCHIVE=/content/source-upload.zip
env: TL_DATA_ARCHIVE=/content/drive/MyDrive/cv-practice/eurosat_rgb.zip
env: TL_DATA_SHA256=d53b32fa50a695af988548b019ab0ff0317e0c2066d54d7671b16b179da66bb8
env: TL_PROJECT_DIR=/content/cv-practice
env: TL_DATA_DIR=/content/data/EuroSAT_RGB
env: TL_OUTPUT_DIR=/content/eurosat-run-01
env: TL_DATASET_VERSION=1


In [5]:
%%bash
set -euo pipefail
test -d /content
test -f "$TL_SOURCE_ARCHIVE" || { echo 'Upload source-upload.zip to Colab first'; exit 1; }
test -f "$TL_DATA_ARCHIVE" || { echo 'Mount Google Drive and check TL_DATA_ARCHIVE first'; exit 1; }
printf '%s  %s\n' "$TL_DATA_SHA256" "$TL_DATA_ARCHIVE" | sha256sum --check
mkdir -p "$TL_PROJECT_DIR" "$(dirname "$TL_DATA_DIR")"
unzip -oq "$TL_SOURCE_ARCHIVE" -d "$TL_PROJECT_DIR"
unzip -oq "$TL_DATA_ARCHIVE" -d "$(dirname "$TL_DATA_DIR")"
test -f "$TL_PROJECT_DIR/tasks/transfer_learning/main.py"
test -d "$TL_DATA_DIR/AnnualCrop"
test -d "$TL_DATA_DIR/SeaLake"


/content/drive/MyDrive/cv-practice/eurosat_rgb.zip: OK


In [6]:
%%bash
set -euo pipefail
"$TL_PYTHON" -m pip install "numpy>=2,<3" "Pillow>=10,<13" "matplotlib>=3.9,<4"
"$TL_PYTHON" -c "import torch, torchvision; print('torch:', torch.__version__, 'torchvision:', torchvision.__version__, 'CUDA:', torch.version.cuda); assert torch.cuda.is_available(), 'CUDA unavailable'; print(torch.cuda.get_device_name(0))"


torch: 2.14.1+cu130 torchvision: 0.29.1+cu130 CUDA: 13.0
Tesla T4


In [7]:
%%bash
set -euo pipefail
cd "$TL_PROJECT_DIR"
"$TL_PYTHON" -u -m tasks.transfer_learning.main \
  --data-dir "$TL_DATA_DIR" \
  --output-dir "$TL_OUTPUT_DIR" \
  --dataset-version "$TL_DATASET_VERSION" \
  --device cuda --seed 42 \
  --head-epochs 5 --finetune-epochs 10 \
  --head-lr 0.001 --finetune-lr 0.0001 \
  --batch-size 64 --workers 2 --weight-decay 0.0001


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
head_only 1/5: train_acc=0.8512 val_acc=0.9126 val_loss=0.3091
head_only 2/5: train_acc=0.9203 val_acc=0.9257 val_loss=0.2367
head_only 3/5: train_acc=0.9298 val_acc=0.9311 val_loss=0.2117
head_only 4/5: train_acc=0.9347 val_acc=0.9304 val_loss=0.2017
head_only 5/5: train_acc=0.9374 val_acc=0.9368 val_loss=0.1871
fine_tuned 1/10: train_acc=0.9378 val_acc=0.9560 val_loss=0.1382
fine_tuned 2/10: train_acc=0.9678 val_acc=0.9635 val_loss=0.1152
fine_tuned 3/10: train_acc=0.9740 val_acc=0.9696 val_loss=0.0930
fine_tuned 4/10: train_acc=0.9794 val_acc=0.9632 val_loss=0.1212
fine_tuned 5/10: train_acc=0.9823 val_acc=0.9701 val_loss=0.0908
fine_tuned 6/10: train_acc=0.9839 val_acc=0.9662 val_loss=0.0999
fine_tuned 7/10: train_acc=0.9857 val_acc=0.9716 val_loss=0.0941
fine_tuned 8/10: train_acc=0.9883 val_acc=0.9723 val_loss=0.0879
fine_tuned 9/10: train_acc=0.989

100%|██████████| 44.7M/44.7M [00:00<00:00, 112MB/s] 


In [8]:
%%bash
set -euo pipefail
test -f "$TL_OUTPUT_DIR/completed.json"
cd "$(dirname "$TL_OUTPUT_DIR")"
zip -qr "$TL_OUTPUT_DIR.zip" "$(basename "$TL_OUTPUT_DIR")"
printf 'Download from Colab Contents: %s.zip\n' "$TL_OUTPUT_DIR"


Download from Colab Contents: /content/eurosat-run-01.zip


In [9]:
!cp /content/eurosat-run-01.zip /content/drive/MyDrive/cv-practice/